# X-Ray AI Training Project
### Database Download, editing and organizing

In [2]:
import os
import shutil
import random
from google.colab import userdata

# =====================================================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# =====================================================================
# Retrieve credentials securely from Colab's Secret Manager
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("📥 Downloading dataset via Kaggle API...")
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia > /dev/null
!unzip -q chest-xray-pneumonia.zip -d dataset/
print("✅ Dataset downloaded and extracted.")

# =====================================================================
# 2. DATASET RESTRUCTURING (FIXING ORIGINAL SPLIT)
# =====================================================================
# The original dataset has a flawed validation split (only 16 images).
# We merge train/val and recreate a robust 80/20 split.
base_dir = '/content/dataset/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
classes = ['NORMAL', 'PNEUMONIA']

print("🔄 Restructuring data for an 80/20 train/validation split...")

for cls in classes:
    train_imgs = os.listdir(os.path.join(train_dir, cls))
    val_imgs = os.listdir(os.path.join(val_dir, cls))
    all_imgs = train_imgs + val_imgs
    random.shuffle(all_imgs)

    split_idx = int(len(all_imgs) * 0.8)
    new_train, new_val = all_imgs[:split_idx], all_imgs[split_idx:]

    # Move files to proper directories
    for img in val_imgs:
        shutil.move(os.path.join(val_dir, cls, img), os.path.join(train_dir, cls, img))
    for img in new_val:
        shutil.move(os.path.join(train_dir, cls, img), os.path.join(val_dir, cls, img))

print("✅ Data splits fixed.")

📥 Downloading dataset via Kaggle API...
100% 2.29G/2.29G [00:51<00:00, 47.8MB/s]
✅ Dataset downloaded and extracted.
🔄 Restructuring data for an 80/20 train/validation split...
✅ Data splits fixed.


### Model Setup and data loaders

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# =====================================================================
# 3. DATALOADERS & AUGMENTATION (RUN ONCE PER SESSION)
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Prevent overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32
train_loader = DataLoader(datasets.ImageFolder(train_dir, transform=train_transforms), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=val_transforms), batch_size=batch_size, shuffle=False)

print("✅ DataLoaders created successfully. No need to run this cell again for new experiments.")

✅ DataLoaders created successfully. No need to run this cell again for new experiments.


### Train-Loader and Val-Loader

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Definir Transformaciones (Data Augmentation y Normalización para ResNet)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Pequeña rotación para evitar overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Cargar Datasets desde las carpetas
train_dir = '/content/dataset/chest_xray/train'
val_dir = '/content/dataset/chest_xray/val'

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

# 3. Crear los DataLoaders (¡Esto es lo que faltaba!)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ DataLoaders creados con éxito.")
print(f"Lotes de entrenamiento: {len(train_loader)} (de {batch_size} imágenes cada uno)")
print(f"Lotes de validación: {len(val_loader)}")

✅ DataLoaders creados con éxito.
Lotes de entrenamiento: 131 (de 32 imágenes cada uno)
Lotes de validación: 33


### Experiment parameters

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# =====================================================================
# 🎛️ EXPERIMENT I1: DENSENET121 BASELINE CONTROL PANEL
# =====================================================================
EXPERIMENT_NAME = "expI1_densenet_baseline"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🧪 Preparing: {EXPERIMENT_NAME} | Architecture: DenseNet121")

# 1. Load the Gold Standard Medical Architecture (DenseNet121)
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

# 2. Freeze base layers (Transfer Learning)
for param in model.parameters():
    param.requires_grad = False

# 3. Reset the classification head
# NOTE: DenseNet uses '.classifier' instead of '.fc'
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2).to(device)
model = model.to(device)

# 4. Initialize Loss and Optimizer (Neutral weights for fair baseline comparison)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print("✅ DenseNet121 compiled with standard CrossEntropyLoss. Ready to train!")

🧪 Preparing: expI1_densenet_baseline | Architecture: DenseNet121
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 188MB/s]


✅ DenseNet121 compiled with standard CrossEntropyLoss. Ready to train!


### Training loop

In [16]:
import time
import numpy as np
from sklearn.metrics import confusion_matrix
from google.colab import files

# =====================================================================
# 5. TRAINING LOOP WITH BUSINESS METRICS EVALUATION
# =====================================================================
num_epochs = 5
print(f"🚀 Starting training for {num_epochs} epochs...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 20)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        all_preds, all_labels = [], []

        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(dataloader.dataset)

        # Calculate Confusion Matrix for business logic evaluation
        cm = confusion_matrix(all_labels, all_preds)
        tn, fp, fn, tp = cm.ravel()
        accuracy = (tp + tn) / (tp + tn + fp + fn)

        print(f'[{phase.upper()}] Loss: {epoch_loss:.4f} | Accuracy: {accuracy:.4f}')

        if phase == 'val':
            print(f'   🚨 False Negatives (Missed Patients): {fn}')
            print(f'   💸 False Positives (Healthy but flagged): {fp}')
            print(f'   ✅ Correct Predictions: {tp} Pneumonia, {tn} Normal')

total_time = time.time() - start_time
print(f'\n🏁 Training completed in {total_time // 60:.0f}m {total_time % 60:.0f}s')

# Save model artifacts
model_filename = f"{EXPERIMENT_NAME}.pth"
torch.save(model.state_dict(), model_filename)
files.download(model_filename)
print(f"💾 Model weights saved locally as: {model_filename}")


🚀 Starting training for 5 epochs...

Epoch 1/5
--------------------
[TRAIN] Loss: 0.3330 | Accuracy: 0.8640
[VAL] Loss: 0.2388 | Accuracy: 0.9245
   🚨 False Negatives (Missed Patients): 70
   💸 False Positives (Healthy but flagged): 9
   ✅ Correct Predictions: 707 Pneumonia, 261 Normal

Epoch 2/5
--------------------
[TRAIN] Loss: 0.1965 | Accuracy: 0.9228
[VAL] Loss: 0.1498 | Accuracy: 0.9465
   🚨 False Negatives (Missed Patients): 24
   💸 False Positives (Healthy but flagged): 32
   ✅ Correct Predictions: 753 Pneumonia, 238 Normal

Epoch 3/5
--------------------
[TRAIN] Loss: 0.1683 | Accuracy: 0.9384
[VAL] Loss: 0.1411 | Accuracy: 0.9484
   🚨 False Negatives (Missed Patients): 34
   💸 False Positives (Healthy but flagged): 20
   ✅ Correct Predictions: 743 Pneumonia, 250 Normal

Epoch 4/5
--------------------
[TRAIN] Loss: 0.1623 | Accuracy: 0.9381
[VAL] Loss: 0.1659 | Accuracy: 0.9436
   🚨 False Negatives (Missed Patients): 55
   💸 False Positives (Healthy but flagged): 4
   ✅ Corre

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾 Model weights saved locally as: expI1_densenet_baseline.pth
